# Task 6. Xác minh replay idempotent

## Mục tiêu

Task 6 kiểm chứng luồng replay tăng dần ở mức parser core: file Python được parse thành graph events, state SQLite được commit sau khi writer flush thành công, file không đổi được bỏ qua, và file đã sửa sinh diff DELETE/UPSERT để downstream có thể cập nhật idempotent.


## Tiêu chí xác minh

| Tiêu chí | Cách kiểm chứng trong notebook |
|---|---|
| File `.py` mới parse thành công | Tạo fixture `pkg/new_feature.py`, chạy `ProcessFileService.execute`. |
| File không đổi được bỏ qua | Chạy lại cùng file và kỳ vọng `SKIPPED_UNCHANGED`. |
| File đã sửa sinh replay diff | Sửa fixture, chạy `ReplayFileService.execute`, kiểm tra old/new hash và số event diff. |
| Không sinh parser error | Đếm `errors.jsonl` bằng `0`. |
| State không bị trùng | Đọc SQLite `file_state` theo `file_id`, kỳ vọng chỉ một row được overwrite với graph mới. |


## Thiết kế

Notebook dùng một repository fixture tạm trong `workspace/tmp/task6-replay-notebook`. Toàn bộ kiểm chứng đi qua các adapter thật của project: `GitSourceRepository`, `CpgParser`, `ProcessFileService`, `ReplayFileService`, `JsonlEventWriter`, `EventValidator` và `SqliteStateStore`.

Không cần Kafka/Neo4j/MongoDB để kiểm chứng phần này: JSONL writer đại diện cho event stream producer, còn SQLite state chứng minh cơ chế incremental replay của parser core. Task 4 và Task 5 kiểm chứng downstream riêng.


## Chuẩn bị notebook runtime

Cell này chuẩn bị workspace tạm, import service thật và tạo helper nhỏ để việc kiểm chứng ở các cell sau đọc được như một kịch bản tuyến tính.


In [1]:
from pathlib import Path
import json
import shutil
import sqlite3
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "lab04-book":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from application.services.process_file import ProcessFileService
from application.services.replay_file import ReplayFileService
from domain.models import SourceFile
from infrastructure.filesystem.git_source_repository import GitSourceRepository
from infrastructure.messaging.event_validator import EventValidator
from infrastructure.messaging.jsonl_event_writer import JsonlEventWriter
from infrastructure.state.sqlite_state_store import SqliteStateStore
from parsing.cpg_parser import CpgParser
from parsing.identifiers import IdentifierGenerator

verification_dir = PROJECT_ROOT / "workspace" / "tmp" / "task6-replay-notebook"
source_root = verification_dir / "source"
output_dir = verification_dir / "events"
state_db = verification_dir / "state.sqlite3"
repo_id = "local/task6-replay-notebook"
target_file = Path("pkg/new_feature.py")
target_path = source_root / target_file

if verification_dir.exists():
    shutil.rmtree(verification_dir)
target_path.parent.mkdir(parents=True)
output_dir.mkdir(parents=True)

repo = GitSourceRepository(source_root, "", None)
parser = CpgParser(repository_id=repo_id)
state_store = SqliteStateStore(state_db, repo_id)
validator = EventValidator(PROJECT_ROOT / "schemas")
writer = JsonlEventWriter(output_dir)
process_service = ProcessFileService(repo, parser, state_store, validator, writer)
replay_service = ReplayFileService(repo, parser, state_store, process_service, repo_id)


def write_source(source: str) -> None:
    target_path.write_text(source, encoding="utf-8")


def make_source_file() -> SourceFile:
    return SourceFile(
        repository_id=repo_id,
        repository_root=str(source_root),
        relative_path=target_file.as_posix(),
        commit_sha=repo.get_commit_hash(),
        size_bytes=target_path.stat().st_size,
    )


def jsonl_count(filename: str) -> int:
    path = output_dir / filename
    if not path.exists():
        return 0
    return sum(1 for line in path.read_text(encoding="utf-8").splitlines() if line.strip())


def jsonl_counts() -> dict[str, int]:
    return {
        "nodes.jsonl": jsonl_count("nodes.jsonl"),
        "edges.jsonl": jsonl_count("edges.jsonl"),
        "metadata.jsonl": jsonl_count("metadata.jsonl"),
        "errors.jsonl": jsonl_count("errors.jsonl"),
    }


def sqlite_state_summary() -> dict[str, object]:
    file_id = IdentifierGenerator.generate_file_id(repo_id, target_file)
    with sqlite3.connect(state_db) as conn:
        rows = conn.execute(
            "SELECT file_id, file_path, content_hash, node_ids_json, edge_ids_json "
            "FROM file_state WHERE file_id = ?",
            (file_id,),
        ).fetchall()
    row = rows[0] if rows else None
    return {
        "row_count": len(rows),
        "file_path": row[1] if row else None,
        "content_hash": row[2] if row else None,
        "node_count": len(json.loads(row[3])) if row else 0,
        "edge_count": len(json.loads(row[4])) if row else 0,
    }

print("verification_dir=", verification_dir)


verification_dir= C:\Users\huynh\Documents\SOS\MATERIAL\BD\LAB04\workspace\tmp\task6-replay-notebook


## Thực thi

Cell này chạy đủ ba phase: thêm file mới, chạy lại khi chưa đổi, rồi sửa file và replay.


In [2]:
initial_source = """def normalize(value):
    total = value + 1
    return total
"""
modified_source = """def normalize(value):
    total = value + 1
    if total > 10:
        return total * 2
    return total

class Normalizer:
    def apply(self, value):
        return normalize(value)
"""

write_source(initial_source)
first_result = process_service.execute(make_source_file())
after_first_counts = jsonl_counts()

unchanged_result = process_service.execute(make_source_file())
after_unchanged_counts = jsonl_counts()

write_source(modified_source)
replay_result = replay_service.execute(target_file)
after_replay_counts = jsonl_counts()

run_summary = {
    "first": {
        "status": first_result.status.value,
        "node_count": first_result.node_count,
        "edge_count": first_result.edge_count,
        "emitted_event_counts": first_result.emitted_event_counts,
        "jsonl_counts": after_first_counts,
    },
    "unchanged": {
        "status": unchanged_result.status.value,
        "emitted_event_counts": unchanged_result.emitted_event_counts,
        "jsonl_counts": after_unchanged_counts,
    },
    "replay_after_edit": replay_result,
    "after_replay_jsonl_counts": after_replay_counts,
    "sqlite_state": sqlite_state_summary(),
}
print(json.dumps(run_summary, indent=2, ensure_ascii=False))


{
  "first": {
    "status": "SUCCESS",
    "node_count": 19,
    "edge_count": 20,
    "emitted_event_counts": {
      "cpg.nodes": 19,
      "cpg.edges": 20,
      "source.metadata": 1
    },
    "jsonl_counts": {
      "nodes.jsonl": 19,
      "edges.jsonl": 20,
      "metadata.jsonl": 1,
      "errors.jsonl": 0
    }
  },
  "unchanged": {
    "status": "SKIPPED_UNCHANGED",
    "emitted_event_counts": {},
    "jsonl_counts": {
      "nodes.jsonl": 19,
      "edges.jsonl": 20,
      "metadata.jsonl": 1,
      "errors.jsonl": 0
    }
  },
  "replay_after_edit": {
    "file_path": "pkg/new_feature.py",
    "status": "SUCCESS",
    "old_content_hash": "4d5df56dd3faea85b9109f93515e6674e0ee1cabf24c9816bf1b8ffcb4eb09e2",
    "new_content_hash": "abca7550e6e16328180fe82639c4fda16bfec385e680c9968f109b647642d07b",
    "removed_node_count": 3,
    "removed_edge_count": 6,
    "upsert_node_count": 44,
    "upsert_edge_count": 51,
    "error": null
  },
  "after_replay_jsonl_counts": {
    "node

## Xác minh

Cell này biến kết quả chạy thành các điều kiện kiểm chứng rõ ràng. Nếu pipeline local không đạt, notebook sẽ fail ngay tại assertion tương ứng.


In [3]:
assert run_summary["first"]["status"] == "SUCCESS"
assert run_summary["first"]["node_count"] > 0
assert run_summary["first"]["edge_count"] > 0
assert run_summary["first"]["jsonl_counts"]["metadata.jsonl"] == 1

assert run_summary["unchanged"]["status"] == "SKIPPED_UNCHANGED"
assert run_summary["unchanged"]["emitted_event_counts"] == {}
assert run_summary["unchanged"]["jsonl_counts"] == run_summary["first"]["jsonl_counts"]

replay = run_summary["replay_after_edit"]
assert replay["status"] == "SUCCESS"
assert replay["old_content_hash"] != replay["new_content_hash"]
assert replay["removed_node_count"] > 0
assert replay["removed_edge_count"] > 0
assert replay["upsert_node_count"] > run_summary["first"]["node_count"]
assert replay["upsert_edge_count"] > run_summary["first"]["edge_count"]

assert run_summary["after_replay_jsonl_counts"]["errors.jsonl"] == 0
assert run_summary["after_replay_jsonl_counts"]["metadata.jsonl"] == 2
assert run_summary["sqlite_state"]["row_count"] == 1
assert run_summary["sqlite_state"]["node_count"] == replay["upsert_node_count"]
assert run_summary["sqlite_state"]["edge_count"] == replay["upsert_edge_count"]

verification_summary = {
    "add_new_python_file": "passed",
    "skip_unchanged_file": "passed",
    "edit_file_replay_diff": "passed",
    "jsonl_error_count": run_summary["after_replay_jsonl_counts"]["errors.jsonl"],
    "sqlite_state_rows": run_summary["sqlite_state"]["row_count"],
}
print(json.dumps(verification_summary, indent=2, ensure_ascii=False))


{
  "add_new_python_file": "passed",
  "skip_unchanged_file": "passed",
  "edit_file_replay_diff": "passed",
  "jsonl_error_count": 0,
  "sqlite_state_rows": 1
}


## Diễn giải

Kết quả cho thấy parser core xử lý đúng hai tình huống quan trọng của replay tăng dần. Khi file mới xuất hiện, service sinh node/edge/metadata events và commit state ban đầu. Khi chạy lại cùng nội dung, `content_hash`, `parser_version` và `schema_version` khớp với SQLite state nên file được bỏ qua. Khi file thay đổi, replay tạo diff với DELETE/UPSERT events, cập nhật JSONL event stream và overwrite state hiện có theo cùng `file_id`.


## Kết quả

- Thêm file `.py` mới: `SUCCESS`, có node/edge events và metadata event.
- Chạy lại khi file chưa đổi: `SKIPPED_UNCHANGED`, JSONL counts không tăng.
- Sửa file rồi replay: `SUCCESS`, old/new content hash khác nhau và có removed/upsert diff.
- `errors.jsonl=0`, chứng minh parser không gặp lỗi cú pháp trong fixture.
- SQLite `file_state` chỉ có một row theo `file_id`, với node/edge count bằng graph mới sau replay.


## Lệnh xác minh end-to-end

```powershell
jupyter nbconvert --to notebook --execute lab04-book/task6_idempotent_replay.ipynb --inplace --ExecutePreprocessor.timeout=120
```

Khi chạy live pipeline đầy đủ, thay JSONL writer bằng Kafka producer qua CLI:

```powershell
docker compose --env-file .env -f infra/docker-compose.yml -f infra/docker-compose.neo4j.yml up -d kafka neo4j kafka-connect mongodb
uv run lab04 replay-file --file path/to/modified.py --no-dry-run
```


## Reflection

Idempotent replay chỉ hoàn chỉnh khi producer, message broker và downstream storage cùng tuân thủ stable identity. Notebook này kiểm chứng phần producer/parser core: deterministic IDs, skip unchanged files, graph diff và SQLite commit sau writer flush. Neo4j Sink và Spark/MongoDB tiếp tục dùng các stable keys đó để cập nhật downstream mà không tạo duplicate.
